## Step 2 - Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("tensorflow version:", tf.__version__)

## Step 3 - Load the Dataset

two options:
- option A = load from kaggle CSV files (if you downloaded the dataset)
- option B = load directly from keras (no download needed)

use whichever works for you

In [ ]:
# OPTION A - load from kaggle CSV files
# put mnist_train.csv and mnist_test.csv in your project folder

train_df = pd.read_csv('mnist_train.csv')
test_df  = pd.read_csv('mnist_test.csv')

print("train shape:", train_df.shape)   # (60000, 785)
print("test shape: ", test_df.shape)    # (10000, 785)
print()
print("columns: label + 784 pixel columns")
print(train_df.head(2))

In [ ]:
# OPTION B - load directly from keras (simpler)
# comment out OPTION A above if using this

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print("X_train shape:", X_train.shape)   # (60000, 28, 28)
print("X_test shape: ", X_test.shape)    # (10000, 28, 28)

In [ ]:
# if using OPTION A - convert CSV to image arrays
# skip this cell if using OPTION B

y_train = train_df['label'].values
X_train = train_df.drop('label', axis=1).values.reshape(-1, 28, 28)

y_test  = test_df['label'].values
X_test  = test_df.drop('label', axis=1).values.reshape(-1, 28, 28)

print("X_train shape after reshape:", X_train.shape)   # (60000, 28, 28)
print("X_test shape after reshape: ", X_test.shape)    # (10000, 28, 28)

## Step 4 - EDA (Look at the Data)

In [ ]:
# show some images
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"label: {y_train[i]}")
    plt.axis('off')
plt.suptitle("Sample MNIST Images")
plt.tight_layout()
plt.show()

In [ ]:
# class distribution
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"digit {u}: {c} images")

# all should be around 6000 - balanced dataset

In [ ]:
# check pixel range
print("min pixel:", X_train.min())   # 0
print("max pixel:", X_train.max())   # 255

## Step 5 - Preprocessing

normalize pixels from 0-255 to 0-1  
add channel dimension for CNN (28,28) → (28,28,1)

In [ ]:
# normalize to 0-1
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

# add channel dimension - needed for Conv2D layers
# (60000, 28, 28) → (60000, 28, 28, 1)
X_train = np.expand_dims(X_train, axis=-1)
X_test  = np.expand_dims(X_test,  axis=-1)

print("X_train shape:", X_train.shape)   # (60000, 28, 28, 1)
print("X_test shape: ", X_test.shape)    # (10000, 28, 28, 1)
print("max pixel after normalization:", X_train.max())   # 1.0

## Step 6 - Add Noise to Images

we add gaussian noise to the images  
noisy image = input to the model  
clean image = target output of the model  
model learns to remove the noise

In [ ]:
noise_factor = 0.3   # how much noise to add - try 0.1 to 0.5

# add random gaussian noise
X_train_noisy = X_train + noise_factor * np.random.normal(size=X_train.shape)
X_test_noisy  = X_test  + noise_factor * np.random.normal(size=X_test.shape)

# clip values to stay between 0 and 1
X_train_noisy = np.clip(X_train_noisy, 0.0, 1.0)
X_test_noisy  = np.clip(X_test_noisy,  0.0, 1.0)

print("noisy train shape:", X_train_noisy.shape)
print("noisy test shape: ", X_test_noisy.shape)

In [ ]:
# compare clean vs noisy images
plt.figure(figsize=(12, 4))
for i in range(8):
    # clean image
    plt.subplot(2, 8, i+1)
    plt.imshow(X_train[i].reshape(28,28), cmap='gray')
    plt.title("clean")
    plt.axis('off')

    # noisy image
    plt.subplot(2, 8, i+9)
    plt.imshow(X_train_noisy[i].reshape(28,28), cmap='gray')
    plt.title("noisy")
    plt.axis('off')

plt.suptitle("Clean vs Noisy Images")
plt.tight_layout()
plt.show()

## Step 7 - Simple Autoencoder (Dense Layers)

encoder = 784 → 128 → 64 (bottleneck)  
decoder = 64 → 128 → 784  
bottleneck = the compressed representation of the image

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Reshape

# ENCODER
input_img = Input(shape=(28, 28, 1))
x = Flatten()(input_img)              # 28x28x1 = 784
x = Dense(128, activation='relu')(x)
encoded = Dense(64, activation='relu')(x)   # bottleneck - 64 values

# DECODER
x = Dense(128, activation='relu')(encoded)
x = Dense(784, activation='sigmoid')(x)    # sigmoid keeps values 0-1
decoded = Reshape((28, 28, 1))(x)          # reshape back to image

# full autoencoder
simple_ae = Model(input_img, decoded)
simple_ae.summary()

In [ ]:
simple_ae.compile(
    optimizer='adam',
    loss='binary_crossentropy'   # good for pixel values between 0-1
)

# input = noisy image, target = clean image
simple_ae_history = simple_ae.fit(
    X_train_noisy, X_train,   # noisy input → clean output
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

## Step 8 - Convolutional Autoencoder

CNN autoencoder is better for images than dense autoencoder  
encoder uses Conv2D + MaxPooling to compress  
decoder uses Conv2DTranspose to reconstruct (reverse of convolution)

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D

# ENCODER
input_img = Input(shape=(28, 28, 1))

x = Conv2D(32, (3,3), activation='relu', padding='same')(input_img)  # (28,28,32)
x = MaxPooling2D((2,2), padding='same')(x)                           # (14,14,32)
x = Conv2D(16, (3,3), activation='relu', padding='same')(x)          # (14,14,16)
encoded = MaxPooling2D((2,2), padding='same')(x)                     # (7,7,16) bottleneck

# DECODER
x = Conv2D(16, (3,3), activation='relu', padding='same')(encoded)    # (7,7,16)
x = UpSampling2D((2,2))(x)                                          # (14,14,16)
x = Conv2D(32, (3,3), activation='relu', padding='same')(x)          # (14,14,32)
x = UpSampling2D((2,2))(x)                                          # (28,28,32)
decoded = Conv2D(1, (3,3), activation='sigmoid', padding='same')(x)  # (28,28,1)

# full convolutional autoencoder
conv_ae = Model(input_img, decoded)
conv_ae.summary()

In [ ]:
conv_ae.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

conv_ae_history = conv_ae.fit(
    X_train_noisy, X_train,   # noisy input → clean output
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

## Step 9 - Plot Training Loss

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(simple_ae_history.history['loss'],     label='train loss')
plt.plot(simple_ae_history.history['val_loss'], label='val loss')
plt.title("Simple Autoencoder Loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(conv_ae_history.history['loss'],     label='train loss')
plt.plot(conv_ae_history.history['val_loss'], label='val loss')
plt.title("Conv Autoencoder Loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.legend()

plt.tight_layout()
plt.show()

## Step 10 - Denoise Test Images and Visualize Results

feed noisy test images into both models  
compare: noisy input → simple AE output → conv AE output → original clean

In [ ]:
# get denoised outputs
simple_denoised = simple_ae.predict(X_test_noisy)
conv_denoised   = conv_ae.predict(X_test_noisy)

print("simple denoised shape:", simple_denoised.shape)
print("conv denoised shape:  ", conv_denoised.shape)

In [ ]:
# visualize - 4 rows: original, noisy, simple AE output, conv AE output
n = 8   # number of images to show

plt.figure(figsize=(16, 8))

for i in range(n):
    # row 1 - original clean
    plt.subplot(4, n, i+1)
    plt.imshow(X_test[i].reshape(28,28), cmap='gray')
    if i == 0: plt.ylabel("original", fontsize=10)
    plt.axis('off')

    # row 2 - noisy input
    plt.subplot(4, n, i+1+n)
    plt.imshow(X_test_noisy[i].reshape(28,28), cmap='gray')
    if i == 0: plt.ylabel("noisy", fontsize=10)
    plt.axis('off')

    # row 3 - simple AE output
    plt.subplot(4, n, i+1+2*n)
    plt.imshow(simple_denoised[i].reshape(28,28), cmap='gray')
    if i == 0: plt.ylabel("simple AE", fontsize=10)
    plt.axis('off')

    # row 4 - conv AE output
    plt.subplot(4, n, i+1+3*n)
    plt.imshow(conv_denoised[i].reshape(28,28), cmap='gray')
    if i == 0: plt.ylabel("conv AE", fontsize=10)
    plt.axis('off')

plt.suptitle("Original vs Noisy vs Simple AE vs Conv AE")
plt.tight_layout()
plt.show()

# conv AE should produce sharper cleaner images than simple AE

## Step 11 - Evaluate with PSNR and MSE

MSE = mean squared error - lower is better  
PSNR = peak signal to noise ratio - higher is better  
these tell us how good the denoising is numerically

In [ ]:
from sklearn.metrics import mean_squared_error

def calc_psnr(original, reconstructed):
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * np.log10(1.0 / np.sqrt(mse))   # max pixel = 1.0

# MSE
mse_noisy  = np.mean((X_test - X_test_noisy)  ** 2)
mse_simple = np.mean((X_test - simple_denoised) ** 2)
mse_conv   = np.mean((X_test - conv_denoised)  ** 2)

# PSNR
psnr_noisy  = calc_psnr(X_test, X_test_noisy)
psnr_simple = calc_psnr(X_test, simple_denoised)
psnr_conv   = calc_psnr(X_test, conv_denoised)

print("=" * 45)
print(f"{'Model':<20} {'MSE':>10} {'PSNR':>10}")
print("=" * 45)
print(f"{'Noisy (no model)':<20} {mse_noisy:>10.4f} {psnr_noisy:>10.2f}")
print(f"{'Simple AE':<20} {mse_simple:>10.4f} {psnr_simple:>10.2f}")
print(f"{'Conv AE':<20} {mse_conv:>10.4f} {psnr_conv:>10.2f}")
print("=" * 45)
print("lower MSE = better | higher PSNR = better")

## Step 12 - Visualize the Bottleneck (Encoded Representation)

encoder part alone - see what the compressed representation looks like  
this is what the model learns to keep as important information

In [ ]:
# extract just the encoder from conv autoencoder
encoder_model = Model(inputs=conv_ae.input,
                      outputs=conv_ae.get_layer(index=6).output)  # bottleneck layer

# encode some test images
encoded_imgs = encoder_model.predict(X_test[:10])

print("original image shape:", X_test[0].shape)       # (28, 28, 1)
print("encoded image shape: ", encoded_imgs[0].shape) # (7, 7, 16) - much smaller
print("compression ratio:   ", 28*28*1, "→", 7*7*16, "pixels")

## Step 13 - Try Different Noise Levels

see how the model performs when noise is very high vs low

In [ ]:
noise_levels = [0.1, 0.3, 0.5, 0.7]

plt.figure(figsize=(16, 8))
row = 0

for noise in noise_levels:
    # add noise
    X_noisy_temp = X_test + noise * np.random.normal(size=X_test.shape)
    X_noisy_temp = np.clip(X_noisy_temp, 0.0, 1.0)

    # denoise with conv AE
    denoised_temp = conv_ae.predict(X_noisy_temp[:8], verbose=0)

    for i in range(8):
        # noisy
        plt.subplot(len(noise_levels)*2, 8, row*8 + i + 1)
        plt.imshow(X_noisy_temp[i].reshape(28,28), cmap='gray')
        if i == 0: plt.ylabel(f"noisy\n{noise}", fontsize=8)
        plt.axis('off')

        # denoised
        plt.subplot(len(noise_levels)*2, 8, (row+1)*8 + i + 1)
        plt.imshow(denoised_temp[i].reshape(28,28), cmap='gray')
        if i == 0: plt.ylabel("denoised", fontsize=8)
        plt.axis('off')

    row += 2

plt.suptitle("Conv AE performance at different noise levels")
plt.tight_layout()
plt.show()

## Step 14 - Save the Models

In [ ]:
simple_ae.save('simple_autoencoder.h5')
conv_ae.save('conv_autoencoder.h5')

print("both models saved")

# to load later:
# from tensorflow.keras.models import load_model
# model = load_model('conv_autoencoder.h5')

## Theory Notes

**what is an autoencoder**
- a type of neural network that learns to compress and reconstruct data
- encoder = input → small representation (bottleneck)
- decoder = small representation → reconstructed output
- trained to make output as close to input as possible

**what is a denoising autoencoder**
- input = noisy/corrupted image
- output = clean image
- model learns to ignore noise and keep important features

**bottleneck**
- the compressed middle layer of the autoencoder
- forces the model to learn only the most important features
- example: 784 pixels → 64 values → 784 pixels

**Simple AE vs Conv AE**
- simple AE uses dense layers - flattens the image, loses spatial info
- conv AE uses Conv2D - keeps spatial info - better for images
- conv AE always gives sharper cleaner output

**UpSampling2D**
- reverse of MaxPooling
- doubles the image size
- used in decoder to go from small back to big

**MSE (Mean Squared Error)**
- average of squared differences between original and reconstructed
- lower = better reconstruction

**PSNR (Peak Signal to Noise Ratio)**
- measures quality of reconstruction in dB
- higher = better quality
- above 30 dB is considered good

**binary_crossentropy as loss**
- used because pixel values are between 0 and 1 after normalization
- works better than MSE for image reconstruction

**noise_factor**
- 0.1 = light noise - easy for model
- 0.3 = medium noise - good training point
- 0.5+ = heavy noise - harder for model to recover

---

## Expected Results

| model | MSE | PSNR | image quality |
|---|---|---|---|
| noisy (no model) | highest | lowest | very noisy |
| Simple AE | medium | medium | slightly blurry |
| Conv AE | lowest | highest | sharp and clean |

conv AE will always outperform simple AE for image denoising